# {{PROJECT_NAME}}

KFP v2 fine-tuning pipeline on the Miramar platform.

**Development workflow:**
1. Write step logic in the `@dsl.component` cells below
2. Wire steps in the **Pipeline** cell
3. Save (`Ctrl+S`), then run **Build → pipeline.py**
4. Compile and submit in **Compile & Submit**

## Step Development

Write your pipeline step logic directly inside each `@dsl.component` function body.

- **Imports must be inside the function** — KFP runs each component in an isolated container
- Set `base_image` and `packages_to_install` on `@dsl.component` to match your step’s runtime
- GPU steps (train, merge_adapter, quantize) use `pytorch/pytorch:2.5.1-cuda12.4-cudnn9-devel`; CPU steps use `python:3.11-slim`

In [ ]:
from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Artifact

### prepare_data

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["datasets", "huggingface_hub"],
)
def prepare_data(output_data: Output[Dataset]):
    from datasets import load_dataset
    import json, pathlib
    # TODO: load dataset from HF hub or GCS, format as {"instruction": ..., "response": ...}
    # dataset = load_dataset("your-org/your-dataset", split="train")
    # records = [{"instruction": r["input"], "response": r["output"]} for r in dataset]
    # pathlib.Path(output_data.path).write_text(json.dumps(records))
    pathlib.Path(output_data.path).write_text("[]")
    print(f"prepare_data done — {output_data.path}")

### train

In [ ]:
@dsl.component(
    base_image="pytorch/pytorch:2.5.1-cuda12.4-cudnn9-devel",
    packages_to_install=["transformers", "peft", "accelerate", "datasets"],
)
def train(input_data: Input[Dataset], output_model: Output[Model]):
    import json, pathlib
    from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
    from peft import LoraConfig, get_peft_model
    # TODO: set base model
    # BASE_MODEL = "meta-llama/Llama-3.2-1B"
    # tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    # model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype="bfloat16", device_map="auto")
    # lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
    # model = get_peft_model(model, lora_cfg)
    # records = json.loads(pathlib.Path(input_data.path).read_text())
    # ... build dataset, TrainingArguments, Trainer, trainer.train() ...
    # model.save_pretrained(output_model.path)
    # tokenizer.save_pretrained(output_model.path)
    pathlib.Path(output_model.path).mkdir(parents=True, exist_ok=True)
    print(f"train done — {output_model.path}")

### merge_adapter

In [ ]:
@dsl.component(
    base_image="pytorch/pytorch:2.5.1-cuda12.4-cudnn9-devel",
    packages_to_install=["transformers", "peft"],
)
def merge_adapter(input_model: Input[Model], output_model: Output[Model]):
    import pathlib
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    # TODO: set base model (same as train step)
    # BASE_MODEL = "meta-llama/Llama-3.2-1B"
    # base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype="bfloat16", device_map="auto")
    # model = PeftModel.from_pretrained(base, input_model.path)
    # merged = model.merge_and_unload()
    # merged.save_pretrained(output_model.path)
    # AutoTokenizer.from_pretrained(input_model.path).save_pretrained(output_model.path)
    pathlib.Path(output_model.path).mkdir(parents=True, exist_ok=True)
    print(f"merge_adapter done — {output_model.path}")

### quantize

In [ ]:
@dsl.component(
    base_image="pytorch/pytorch:2.5.1-cuda12.4-cudnn9-devel",
    packages_to_install=["autoawq", "transformers"],
)
def quantize(input_model: Input[Model], output_model: Output[Model]):
    import pathlib
    from awq import AutoAWQForCausalLM
    from transformers import AutoTokenizer
    # quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}
    # model = AutoAWQForCausalLM.from_pretrained(input_model.path, device_map="auto")
    # tokenizer = AutoTokenizer.from_pretrained(input_model.path)
    # model.quantize(tokenizer, quant_config=quant_config)
    # model.save_quantized(output_model.path)
    # tokenizer.save_pretrained(output_model.path)
    pathlib.Path(output_model.path).mkdir(parents=True, exist_ok=True)
    print(f"quantize done — {output_model.path}")

### evaluate

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["transformers", "torch", "evaluate", "datasets"],
)
def evaluate(input_model: Input[Model], output_model: Output[Model]):
    import json, pathlib, shutil
    # from transformers import pipeline as hf_pipeline
    # import evaluate as hf_evaluate
    # pipe = hf_pipeline("text-generation", model=input_model.path, device_map="auto")
    # bleu = hf_evaluate.load("sacrebleu")
    # TODO: run inference on test samples, compute metrics, print/assert thresholds
    # results = {"bleu": ..., "samples_evaluated": ...}
    # print(json.dumps(results, indent=2))
    shutil.copytree(input_model.path, output_model.path, dirs_exist_ok=True)
    print(f"evaluate done — {output_model.path}")

### push_to_gcs

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["google-cloud-storage"],
)
def push_to_gcs(input_model: Input[Model], output_artifact: Output[Artifact]):
    import os, pathlib
    from google.cloud import storage
    # GCS_BUCKET = os.environ.get("GCS_BUCKET", "your-bucket")
    # GCS_PREFIX = os.environ.get("GCS_PREFIX", "models/{{PROJECT_NAME}}")
    # client = storage.Client()
    # bucket = client.bucket(GCS_BUCKET)
    # for f in pathlib.Path(input_model.path).rglob("*"):
    #     if f.is_file():
    #         blob = bucket.blob(f"{GCS_PREFIX}/{f.relative_to(input_model.path)}")
    #         blob.upload_from_filename(str(f))
    # gcs_uri = f"gs://{GCS_BUCKET}/{GCS_PREFIX}"
    # pathlib.Path(output_artifact.path).write_text(gcs_uri)
    pathlib.Path(output_artifact.path).write_text("gs://placeholder")
    print(f"push_to_gcs done — {output_artifact.path}")

### deploy

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["requests"],
)
def deploy(input_artifact: Input[Artifact]):
    import pathlib, requests
    gcs_uri = pathlib.Path(input_artifact.path).read_text().strip()
    # OLLAMA_HOST = "http://localhost:11434"
    # model_tag = gcs_uri.rsplit("/", 1)[-1]
    # resp = requests.post(f"{OLLAMA_HOST}/api/pull", json={"name": model_tag})
    # resp.raise_for_status()
    print(f"deploy done — artifact: {gcs_uri}")

### Pipeline

Wire the step tasks together. Access outputs via `.output` (single unnamed output) or `.outputs["name"]` (named outputs).

In [ ]:
@dsl.pipeline(name="{{PROJECT_NAME}}")
def pipeline():
    t1 = prepare_data()
    t2 = train(input_data=t1.outputs["output_data"])
    t3 = merge_adapter(input_model=t2.outputs["output_model"])
    t4 = quantize(input_model=t3.outputs["output_model"])
    t5 = evaluate(input_model=t4.outputs["output_model"])
    t6 = push_to_gcs(input_model=t5.outputs["output_model"])
    deploy(input_artifact=t6.outputs["output_artifact"])

## Build → `pipeline.py`

Save the notebook first (`Ctrl+S`), then run this cell.

In [ ]:
import json, pathlib, re

def build_pipeline(notebook_path="notebook.ipynb"):
    nb = json.loads(pathlib.Path(notebook_path).read_text())
    step_srcs, pipeline_src = [], None
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        tags = cell.get("metadata", {}).get("tags", [])
        src = "".join(cell["source"])
        if "kfp_step" in tags:
            step_srcs.append(src)
        elif "kfp_pipeline" in tags:
            pipeline_src = src
    if not step_srcs:
        raise RuntimeError("No cells tagged 'kfp_step' found.")
    if pipeline_src is None:
        raise RuntimeError("No cell tagged 'kfp_pipeline' found.")
    names = [m.group(1) for src in step_srcs
             for m in [re.search(r"^def (\w+)\(", src, re.MULTILINE)] if m]
    out  = "# Generated by notebook.ipynb \u2014 do not edit manually.\n"
    out += "# Re-run the Build cell to regenerate.\n\n"
    out += "from kfp import dsl\n"
    out += "from kfp.dsl import Input, Output, Dataset, Model, Artifact\n\n\n"
    out += "\n\n\n".join(step_srcs)
    out += "\n\n\n"
    out += pipeline_src
    out += "\n"
    pathlib.Path("pipeline.py").write_text(out)
    print(f"Wrote pipeline.py \u2014 {len(names)} component(s): {', '.join(names)}")

build_pipeline()

## Compile & Submit

In [ ]:
from kfp import compiler
from pipeline import pipeline
compiler.Compiler().compile(pipeline_func=pipeline, package_path='/tmp/pipeline.yaml')
print('Compiled → /tmp/pipeline.yaml')

In [ ]:
# Connect to KFP (requires SSH tunnel: ssh -L 8080:localhost:8080 spark-79b7.local)
import kfp
client = kfp.Client(host='http://localhost:8080')
client.list_pipelines()

In [ ]:
run = client.create_run_from_pipeline_package(
    pipeline_file='/tmp/pipeline.yaml',
    arguments={},
    run_name='notebook-run',
)
print(f'Run ID: {run.run_id}')
print(f'UI: http://localhost:8080/#/runs/details/{run.run_id}')

In [ ]:
import time
run_id = run.run_id  # or paste a run ID here
for _ in range(20):
    r = client.get_run(run_id)
    state = r.state
    print(f'  {state}')
    if state in ('SUCCEEDED', 'FAILED', 'CANCELED'):
        break
    time.sleep(10)